From https://github.com/langchain-ai/langgraph/blob/main/docs/docs/tutorials/lats/lats.ipynb


## Agent LATS

In [ ]:
import os
import pandas as pd
import textwrap
import logging
from termcolor import colored
from langgraph.prebuilt import ToolNode
from dotenv import load_dotenv
from typing import TypedDict
from langchain_ollama import ChatOllama

DEBUG=True

### Logging

In [ ]:

logger = logging.getLogger("my-logger")
if DEBUG: 
    logger.setLevel(logging.DEBUG)
else:
    logger.setLevel(logging.INFO)


if not logger.hasHandlers():
    console_handler = logging.StreamHandler()
    logger.addHandler(console_handler)

In [ ]:
def prettyprint(obj, color='white'):
    wrapper = textwrap.TextWrapper(width=80)
    if isinstance(obj, list):
        text = obj[-1]
        logger.debug(f"### Message {len(obj) - 1} ###")
        if content := getattr(text, 'content', None):
            wrapped_text = wrapper.wrap(str(content))
        else:
            wrapped_text = wrapper.wrap(str(text))
        logger.debug(colored("\n".join(wrapped_text), color=color))
        #logger.debug("\n".join(wrapped_text) + '\n')
    else:
        if content := getattr(obj, 'content', None):
            wrapped_text = wrapper.wrap(str(content))
        else:
            wrapped_text = wrapper.wrap(str(obj))
        logger.debug(colored("\n".join(wrapped_text), color=color))
        #logger.debug("\n".join(wrapped_text) + "\n")

In [ ]:
def pprint(obj, color='white'):
    wrapper = textwrap.TextWrapper(width=82)
    wrapped_text = wrapper.wrap(str(obj))
    print(colored("\n".join(wrapped_text), color=color))

### Instantiate LLM

In [ ]:
llm = ChatOllama(model = "qwen3:32b", temperature=0.0, top_k=2, top_p=0.9, verbose=True)

### Load parameters

In [ ]:
# Load .env file
load_dotenv()
api_key = os.getenv("BRAVE_API_KEY")

In [ ]:
import getpass
import os

def _set_env(var: str) -> None:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Set {var}: ")

os.environ["LANGSMITH_TRACING"] = "true"
_set_env("LANGSMITH_API_KEY")

### Instantiate Search Tool 

In [ ]:
from langchain_community.tools import BraveSearch
search_tool = BraveSearch.from_api_key(api_key=api_key, search_kwargs={"count": 1})
tools = [search_tool]
tool_node = ToolNode(tools=tools)

### Reflection Class

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage

class Reflection(BaseModel):
    reflection: str = Field(
        description="The critique and reflections on the sufficiency, superfluency,"
        " and general quality of the response"
    )
    score: int = Field(
        description="Score from 0-100 on the quality of the candidate response."
    )
    found_solution: bool = Field(
        description="Whether the response has fully solved the question or task."
    )

    def as_message(self):
        return HumanMessage(content=f"Reflection: {self.reflection}\nScore: {self.score}")

    @property
    def normalized_score(self) -> float:
        return self.score / 100.0

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import chain as as_runnable

from langchain_core.runnables import chain as as_runnable
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Reflect and grade the assistant response (candidate) to the user question (question)."
         "Be thorough and do not accept incomplete reposnses. Ask for improvement if the answer is not complete."
         "The score must be an integer number between 0 and 100. " 
         ),
        ("user", "{question}"),
        ("ai", "{candidate}"),
        #("ai", "{data}")
    ]
)

model = llm.with_structured_output(Reflection)
reflection_chain = prompt | model


In [ ]:
from langchain_core.messages import BaseMessage
import math

class Node:
    def __init__(self, messages, response, reflection: Reflection, parent=None):
        self.messages = messages
        self.parent = parent
        self.children = []
        self.reflection = reflection
        self.response = response
        self.value = 0
        self.visits = 0
        self._is_solved = (reflection.found_solution and reflection.score > 5)
        if self._is_solved:
            self._mark_tree_as_solved()

    @property
    def is_solved(self):
        """If any solutions exist, we can end the search."""
        return self._is_solved

    @property
    def height(self) -> int:
        """Check for how far we've rolled out the tree."""
        if self.children:
            return 1 + max([child.height for child in self.children])
        return 1

    def upper_confidence_bound(self, exploration_weight=1.0):
        if self.parent is None:
            raise ValueError
        if self.visits == 0:
            return self.value
        average_reward = self.value / self.visits
        exploration_term = math.sqrt(math.log(self.parent.visits) / self.visits)
        return average_reward + exploration_weight * exploration_term

    def get_messages(self, include_reflections: bool = True):
        if include_reflections:
            return self.messages + [self.reflection.as_message()]
        return self.messages

    def get_trajectory(self, include_reflections: bool = True) -> list[BaseMessage]:
        """Get messages representing this search branch."""
        messages = []
        node = self
        while node:
            messages.extend(
                node.get_messages(include_reflections=include_reflections)[::-1]
            )
            node = node.parent
        # Reverse the final back-tracked trajectory to return in the correct order
        return messages[::-1]  # root solution, reflection, child 1, ...

    def _mark_tree_as_solved(self):
        parent = self.parent
        while parent:
            parent._is_solved = True
            parent = parent.parent

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from typing import Annotated

class TreeState(TypedDict):
    root: Node
    input: str
    response: str
    final_response: str
    messages: Annotated[list, add_messages]

In [ ]:
llm.bind_tools(tools=tools)llm.bind_tools(tools=tools)

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage

def generate_initial_response(state: TreeState):
    """Generate the initial candidate response"""
    pprint("---- Generate Initial Response ----")
    query = state["input"]
    ai_msg = initial_answer_chain.invoke({"input": query})
    messages = [ai_msg.content]

    parsed = []
    for tool_call in ai_msg.tool_calls:
        d = {}
        d['name'] = tool_call['name']
        d['args'] = tool_call['args']
        d['id'] = tool_call['id']
        parsed.append(d)

    for r in parsed:
        tool_msg = tool_node.invoke(
            {
                "messages": [
                    AIMessage(
                        content="",
                        tool_calls=[
                            {"name": r["name"], "args": r["args"], "id": r["id"]}
                        ],
                    )
                ]
            }
        )
        messages.extend(tool_msg['messages'])

    response = answer_chain.invoke(messages)
    pprint('### Initial Response ###')
    pprint(response)
    messages.append(response)
    reflection = reflection_chain.invoke({"question": query, "candidate": messages})
    pprint('\n\n') 
    pprint('### Reflection ###')
    pprint(reflection, color="green")
    state['messages'] = messages
    root = Node(messages, response=response, reflection=reflection)
    state['response'] = response

    return {
        **state,
        "root": root,
    }

In [ ]:
initial_response = generate_initial_response({"input": "Write a research report on lithium pollution."})

In [ ]:
for i, msg in enumerate(initial_response['messages']):
    print(f'---Message {i}---') 
    pprint(msg)

In [ ]:
initial_response['root'].response

In [ ]:
initial_response['root'].messages

In [ ]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an AI assistant."
        " Generate a response for the input question expanding on the messages and tool calls generated by the previous assistant"
        " Generate some variability in your search queries. "
        " Call the tool again to further reasearch for your response."
        " If asked to generate a report, generate it using formal language. Do not make unnecessary comments."),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="messages", optional=True),
    ]
)

expansion_chain = prompt_template | llm.bind_tools(tools=tools)

In [ ]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an AI assistant."
        " Generate a response for the input question based on messages generated by the previous agent. "
        " Expand on the answer give by the previous agent. "
        " If asked to generate a report, generate it using formal language. Do not make unnecessary comments."),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="messages", optional=True),
    ]
)

response_chain = prompt_template | llm

In [ ]:
from langchain_core.prompt_values import ChatPromptValue
from langchain_core.runnables import RunnableConfig

def generate_candidates(state: TreeState):

    n = 2
    all_messages = []
    query = state['input']
    messages = state['messages'] 
    for i in range(n):
        print('### AI MSG ###')
        ai_msg = expansion_chain.invoke({"input": query})
        #ai_msg = expansion_chain.invoke({"input": query, "messages": state['messages']})
        messages.append(ai_msg.content)
        prettyprint(ai_msg)

        parsed = []
        for tool_call in ai_msg.tool_calls:
            d = {}
            d['name'] = tool_call['name']
            d['args'] = tool_call['args']
            d['id'] = tool_call['id']
            parsed.append(d)
        print('### Parsed ###')
        pprint(parsed)
        # tool_responses = []
        for r in parsed:
            tool_msg = tool_node.invoke(
                {
                    "messages": [
                        AIMessage(
                            content="",
                            tool_calls=[
                                {"name": r["name"], "args": r["args"], "id": r["id"]}
                            ],
                        )
                    ]
                }
            )
            messages.extend(tool_msg['messages'])


        print('### Response ###')
        response = response_chain.invoke({'input': query, 'messages': messages})
        prettyprint(response)
        print(
            '\n\n'
        )
        messages.append(response)

        all_messages.append(messages)

    return all_messages


In [ ]:
res = generate_candidates(initial_response)

In [ ]:
len(res)

In [ ]:
for msg in res[0]:
    prettyprint(msg)

In [ ]:
def select(root: Node) -> dict:
    """Starting from the root node a child node is selected at each tree level until a leaf node is reached."""

    print('--- SELECT ---')
    if not root.children:
        print('not children')
        return root

    node = root
    while node.children:
        print(f'children: {node.children}')
        max_child = max(node.children, key=lambda child: child.upper_confidence_bound())
        print('### Max Child ###')
        print(max_child)
        node = max_child

    return node

In [ ]:
from langchain_core.runnables import RunnableConfig
from collections import defaultdict
import textwrap

def expand(state: TreeState):
    """Starting from the best node in the tree, generate N candidates for the next step"""
    root = state["root"]
    best_candidate = select(root)
    messages = best_candidate.get_trajectory(True)

    print('---State Input---')
    print('Question')
    pprint(state['input'])

    new_candidates = generate_candidates(
        {"input": state["input"], "messages": messages}
    )

    child_nodes = []
    for c in new_candidates:
        print("--- Candidate ---")
        # prettyprint(c[1][1])
        response = c[-1]
        reflection = reflection_chain.invoke({"question": state['input'],
                                             "candidate": c})
        
        print('---Response--')
        pprint(response)
        print('\n\n')

        print('---Reflection---')
        pprint(reflection)
        child_nodes.append(
            Node(c, response=response, parent=best_candidate, reflection=reflection)
        )
    best_candidate.children.extend(child_nodes)

    return state


In [ ]:
res = expand(initial_response)

In [ ]:
# wrapper = textwrap.TextWrapper(width=80)
# for i, m in enumerate(res['root'].children[0].get_messages()):
#     print(f'-- Message {i}--')
#     wrapped_text = wrapper.wrap(str(m))
#     print("\n".join(wrapped_text))
#     print('----')
#     print('\n')

In [ ]:
# wrapper = textwrap.TextWrapper(width=80)
# for i, m in enumerate(res['root'].children[1].get_messages()):
#     print(f'-- Message {i}--')
#     wrapped_text = wrapper.wrap(str(m))
#     print("\n".join(wrapped_text))
#     print('----')
#     print('\n')

In [ ]:
from typing import Literal
from langgraph.graph import END, StateGraph, START
from IPython.display import Image


def should_loop(state: TreeState):
    """Determine whether to continue the tree search."""
    root = state["root"]
    if root.is_solved or root.height > 5:
        state['final_response'] = root.get_trajectory()[-1]
        return END
    return "expand"


builder = StateGraph(TreeState)
builder.add_node("start", generate_initial_response)
builder.add_node("expand", expand)
builder.add_edge(START, "start")


builder.add_conditional_edges(
    "start",
    # Either expand/rollout or finish
    should_loop,
    ["expand", END],
)
builder.add_conditional_edges(
    "expand",
    # Either continue to rollout or finish
    should_loop,
    ["expand", END],
)

graph = builder.compile()
Image(graph.get_graph().draw_png())

In [ ]:
question = "Write an essay about Lithium pollution"
response = graph.invoke({"input": question})